# ✈️ Agentic AI-Based Travel Planning Assistant

**Project:** Agentic AI Travel Planner using LangChain + LangGraph + Groq LLaMA  
**Domain:** Travel / Tourism  
**Skills:** Python, LLM Integration, Agentic AI (LangGraph), Prompt Engineering, API Integration, Streamlit  

---

## 🧠 Project Overview

This project builds an **Agentic AI system** that autonomously plans complete travel itineraries.  
The agent receives a user query like *"Plan a 3-day trip to Delhi from Hyderabad within ₹20,000"*  
and automatically calls multiple tools to:

- 🔍 Search for available flights
- 🏨 Recommend hotels
- 🗺️ Discover tourist attractions
- 🌤️ Fetch real-time weather
- 💰 Estimate total budget
- 📋 Generate a complete day-wise itinerary

---

## 📁 Project Structure

```
travel_agent/
├── data/
│   ├── flights.json        # 30 flight records
│   ├── hotels.json         # 40 hotel records
│   └── places.json         # 40 tourist places
├── travel_agent.ipynb      # This notebook
├── app.py                  # Streamlit web app
├── requirements.txt
├── .env                    # API keys (not pushed to GitHub)
└── README.md
```

## 📦 Step 1 — Install Dependencies

In [1]:
!pip install langchain langchain-community langchain-groq langgraph streamlit requests python-dotenv openai -q

## 🔑 Step 2 — Load Environment Variables & API Key

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # Reads API key from .env file

api_key = os.getenv("GROQ_API_KEY")
print("✅ Key loaded:", api_key[:10], "...")

✅ Key loaded: gsk_cVI3tU ...


## 📂 Step 3 — Load Datasets

We use three JSON datasets:
- `flights.json` — 30 flight records with airline, route, time, and price
- `hotels.json` — 40 hotel records with city, stars, price, and amenities
- `places.json` — 40 tourist attractions with city, type, and rating

In [3]:
import json
import requests

with open("data/flights.json") as f:
    flights_data = json.load(f)

with open("data/hotels.json") as f:
    hotels_data = json.load(f)

with open("data/places.json") as f:
    places_data = json.load(f)

print("✅ Flights loaded:", len(flights_data), "records")
print("✅ Hotels loaded: ", len(hotels_data), "records")
print("✅ Places loaded: ", len(places_data), "records")

print("\n--- Sample Records ---")
print("Flight:", flights_data[0])
print("Hotel: ", hotels_data[0])
print("Place: ", places_data[0])

✅ Flights loaded: 30 records
✅ Hotels loaded:  40 records
✅ Places loaded:  40 records

--- Sample Records ---
Flight: {'flight_id': 'FL0001', 'airline': 'IndiGo', 'from': 'Hyderabad', 'to': 'Delhi', 'departure_time': '2025-01-04T11:32:00', 'arrival_time': '2025-01-04T15:32:00', 'price': 2907}
Hotel:  {'hotel_id': 'HOT0001', 'name': 'Grand Palace Hotel', 'city': 'Delhi', 'stars': 4, 'price_per_night': 3897, 'amenities': ['wifi', 'pool']}
Place:  {'place_id': 'PLC0001', 'name': 'Famous Fort', 'city': 'Delhi', 'type': 'lake', 'rating': 4.6}


## 🛠️ Step 4 — Build LangChain Tools

Each tool is a Python function decorated with `@tool`.  
This tells the LangGraph agent that it can call these functions autonomously.

We build **5 tools**:
1. Flight Search Tool
2. Hotel Recommendation Tool
3. Places Discovery Tool
4. Weather Lookup Tool
5. Budget Estimation Tool

In [4]:
from langchain.tools import tool

### Tool 1 — Flight Search

In [5]:
@tool
def search_flights(source: str, destination: str) -> str:
    """Search for available flights from source city to destination city.
    Returns cheapest available flight options."""

    results = [
        f for f in flights_data
        if f["from"].lower() == source.lower()
        and f["to"].lower() == destination.lower()
    ]

    if not results:
        return f"No flights found from {source} to {destination}."

    results.sort(key=lambda x: x["price"])  # Sort by cheapest price

    output = f"Available flights from {source} to {destination}:\n"
    for f in results[:3]:  # Top 3 cheapest
        output += (
            f"- {f['airline']} | Rs.{f['price']} | "
            f"Departs: {f['departure_time'][11:16]} | "
            f"Arrives: {f['arrival_time'][11:16]}\n"
        )
    return output

### Tool 2 — Hotel Recommendation

In [6]:
@tool
def recommend_hotels(city: str, max_price: int = 10000) -> str:
    """Recommend top-rated hotels in a given city within a price range.
    Returns hotel name, stars, price per night, and amenities."""

    results = [
        h for h in hotels_data
        if h["city"].lower() == city.lower()
        and h["price_per_night"] <= max_price
    ]

    if not results:
        return f"No hotels found in {city} under Rs.{max_price}/night."

    results.sort(key=lambda x: x["stars"], reverse=True)  # Sort by highest stars

    output = f"Top hotels in {city}:\n"
    for h in results[:3]:
        output += (
            f"- {h['name']} | {h['stars']} stars | "
            f"Rs.{h['price_per_night']}/night | "
            f"Amenities: {', '.join(h['amenities'])}\n"
        )
    return output

### Tool 3 — Places Discovery

In [7]:
@tool
def find_places(city: str) -> str:
    """Find top tourist attractions and places of interest in a city.
    Returns place name, type, and rating."""

    results = [
        p for p in places_data
        if p["city"].lower() == city.lower()
    ]

    if not results:
        return f"No places found in {city}."

    results.sort(key=lambda x: x["rating"], reverse=True)  # Sort by highest rating

    output = f"Top places to visit in {city}:\n"
    for p in results[:6]:  # Top 6 attractions
        output += f"- {p['name']} | Type: {p['type']} | Rating: {p['rating']}\n"
    return output

### Tool 4 — Weather Lookup

Uses the **Open-Meteo API** — completely free, no API key required.  
Returns a 5-day forecast with temperature and weather condition.

In [8]:
@tool
def get_weather(city: str) -> str:
    """Get real-time weather forecast for a city for the next 5 days.
    Uses Open-Meteo free API. No API key required."""

    # Latitude/Longitude coordinates for major Indian cities
    city_coords = {
        "delhi":     (28.6139, 77.2090),
        "mumbai":    (19.0760, 72.8777),
        "goa":       (15.2993, 74.1240),
        "bangalore": (12.9716, 77.5946),
        "hyderabad": (17.3850, 78.4867),
        "chennai":   (13.0827, 80.2707),
        "kolkata":   (22.5726, 88.3639),
        "jaipur":    (26.9124, 75.7873),
        "pune":      (18.5204, 73.8567),
        "ahmedabad": (23.0225, 72.5714),
    }

    coords = city_coords.get(city.lower())
    if not coords:
        return f"Weather data not available for {city}."

    lat, lon = coords
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
        f"&timezone=auto&forecast_days=5"
    )

    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        dates     = data["daily"]["time"]
        max_temps = data["daily"]["temperature_2m_max"]
        min_temps = data["daily"]["temperature_2m_min"]
        rain      = data["daily"]["precipitation_sum"]

        output = f"Weather forecast for {city}:\n"
        for i in range(5):
            condition = "Rainy" if rain[i] > 2 else "Cloudy" if rain[i] > 0 else "Sunny"
            output += (
                f"- {dates[i]}: {condition} | "
                f"Max: {max_temps[i]}C | Min: {min_temps[i]}C\n"
            )
        return output

    except Exception as e:
        return f"Could not fetch weather data: {str(e)}"

### Tool 5 — Budget Estimation

In [9]:
@tool
def estimate_budget(flight_price: int, hotel_price_per_night: int, num_days: int) -> str:
    """Estimate total trip budget based on flight cost, hotel price per night, and number of days.
    Includes estimated food and local transport costs at Rs.800/day."""

    hotel_total    = hotel_price_per_night * num_days
    food_transport = 800 * num_days  # Rs.800/day for food + local travel
    total          = flight_price + hotel_total + food_transport

    output = (
        f"\nBudget Breakdown:\n"
        f"- Flight:              Rs.{flight_price}\n"
        f"- Hotel ({num_days} nights):   Rs.{hotel_total}\n"
        f"- Food & Transport:    Rs.{food_transport}\n"
        f"------------------------------\n"
        f"- Total Estimate:      Rs.{total}\n"
    )
    return output

## ✅ Step 5 — Test All Tools Individually

Before building the agent, verify each tool works correctly in isolation.

In [10]:
print("=" * 50)
print("TOOL TESTS")
print("=" * 50)

print("\n--- Tool 1: Flight Search ---")
print(search_flights.invoke({"source": "Hyderabad", "destination": "Delhi"}))

print("--- Tool 2: Hotel Recommendation ---")
print(recommend_hotels.invoke({"city": "Delhi"}))

print("--- Tool 3: Places Discovery ---")
print(find_places.invoke({"city": "Delhi"}))

print("--- Tool 4: Weather Forecast ---")
print(get_weather.invoke({"city": "Delhi"}))

print("--- Tool 5: Budget Estimation ---")
print(estimate_budget.invoke({"flight_price": 2907, "hotel_price_per_night": 3650, "num_days": 3}))

TOOL TESTS

--- Tool 1: Flight Search ---
Available flights from Hyderabad to Delhi:
- IndiGo | Rs.2907 | Departs: 11:32 | Arrives: 15:32

--- Tool 2: Hotel Recommendation ---
Top hotels in Delhi:
- Comfort Suites | 5 stars | Rs.3650/night | Amenities: gym, breakfast, wifi, parking
- Grand Palace Hotel | 4 stars | Rs.3897/night | Amenities: wifi, pool
- Green Leaf Resort | 4 stars | Rs.6123/night | Amenities: pool, parking

--- Tool 3: Places Discovery ---
Top places to visit in Delhi:
- Famous Fort | Type: lake | Rating: 4.6
- Popular Museum | Type: lake | Rating: 4.5
- Beautiful Temple | Type: temple | Rating: 4.2
- Historic Fort | Type: museum | Rating: 4.2
- Famous Fort | Type: lake | Rating: 3.5

--- Tool 4: Weather Forecast ---
Weather forecast for Delhi:
- 2026-05-16: Sunny | Max: 40.0C | Min: 28.5C
- 2026-05-17: Sunny | Max: 40.8C | Min: 29.2C
- 2026-05-18: Sunny | Max: 42.7C | Min: 29.2C
- 2026-05-19: Sunny | Max: 42.3C | Min: 31.2C
- 2026-05-20: Sunny | Max: 42.7C | Min: 29.0

## 🤖 Step 6 — Initialize the LLM

We use **Groq's LLaMA 3.3 70B** model — fast and free.  
`temperature=0` ensures consistent, deterministic responses.

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

print("✅ LLM ready — Groq LLaMA 3.3 70B")

✅ LLM ready — Groq LLaMA 3.3 70B


## 🧩 Step 7 — Create the ReAct Agent

### What is a ReAct Agent?

**ReAct = Reasoning + Acting**  
The agent alternates between:
- **Thinking** — deciding which tool to call next
- **Acting** — calling the tool and observing the result

This loop continues until the agent has enough information to produce a final answer.

We use `create_react_agent` from **LangGraph** — the modern standard for building tool-calling agents.

In [12]:
from langgraph.prebuilt import create_react_agent

# Bundle all 5 tools
tools = [search_flights, recommend_hotels, find_places, get_weather, estimate_budget]

# System prompt — instructs the agent on how to behave
system_prompt = """You are an expert AI travel planning assistant.

When a user asks you to plan a trip, use the available tools in this order:
1. search_flights     - find flight from source to destination
2. recommend_hotels   - find hotels at destination
3. find_places        - find tourist attractions at destination
4. get_weather        - get weather forecast for destination
5. estimate_budget    - calculate total trip cost using flight price, hotel price per night, and number of days

After collecting all information, generate a complete structured itinerary with:
- Trip Summary
- Flight Selected (cheapest option)
- Hotel Recommendation (highest rated)
- Day-wise Itinerary (distribute places evenly across days)
- Weather for each day
- Budget Breakdown
- Total Estimated Cost

Always use Rs. for currency. Be specific and well-organized."""

# Create the ReAct agent using LangGraph
agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

print("✅ ReAct Agent ready")
print(f"   Tools registered: {[t.name for t in tools]}")

✅ ReAct Agent ready
   Tools registered: ['search_flights', 'recommend_hotels', 'find_places', 'get_weather', 'estimate_budget']


C:\Users\Dell\AppData\Local\Temp\ipykernel_8028\2860028665.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=system_prompt)


## 🚀 Step 8 — Run the Agent

The agent will autonomously:
1. Parse the user's travel query
2. Call each tool in sequence
3. Combine all results
4. Generate a complete structured itinerary

In [13]:
# User query
user_query = "Plan a 3-day trip to Delhi from Hyderabad. My budget is around Rs.20,000."

print("User Query:", user_query)
print("\nAgent is working...\n")

response = agent_executor.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

print("\n" + "=" * 60)
print("           FINAL ITINERARY")
print("=" * 60)
print(response["messages"][-1].content)

User Query: Plan a 3-day trip to Delhi from Hyderabad. My budget is around Rs.20,000.

Agent is working...


           FINAL ITINERARY
**Trip Summary**
You're planning a 3-day trip to Delhi from Hyderabad with a budget of Rs.20,000. 

**Flight Selected**
The cheapest available flight option from Hyderabad to Delhi is IndiGo, which costs Rs.2907.

**Hotel Recommendation**
The top-rated hotel in Delhi is Comfort Suites, which has 5 stars and costs Rs.3650 per night.

**Day-wise Itinerary**
Here's a suggested day-wise itinerary for your trip:
- Day 1: Visit Famous Fort and Popular Museum
- Day 2: Explore Beautiful Temple and Historic Fort
- Day 3: Visit Famous Fort

**Weather for each day**
The weather forecast for Delhi is as follows:
- Day 1: Sunny | Max: 40.0C | Min: 28.5C
- Day 2: Sunny | Max: 40.8C | Min: 29.2C
- Day 3: Sunny | Max: 42.7C | Min: 29.2C

**Budget Breakdown**
The estimated total trip budget is Rs.13400, which includes:
- Flight: Rs.2907
- Hotel (3 nights): Rs.10950
- F

## 🌐 Step 9 — Streamlit Web Application

The full interactive UI is available in `app.py`.  

To launch it, run this command in your terminal:

```bash
streamlit run app.py
```

The app lets users input source city, destination, number of days, and budget — then generates the full itinerary with one click.

---

## 📊 Project Summary

| Component | Technology |
|---|---|
| LLM | Groq LLaMA 3.3 70B (free) |
| Agent Framework | LangGraph ReAct Agent |
| Tool Integration | LangChain `@tool` decorator |
| Weather API | Open-Meteo (free, no key needed) |
| UI | Streamlit |
| Data | JSON datasets (flights, hotels, places) |
| Language | Python 3.x |